# Set up library

In [ ]:
!pip install autolrtuner

In [ ]:
import numpy as np
import pandas as pd
import librosa
from pathlib import Path
import matplotlib.pyplot as plt
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.losses import CategoricalCrossentropy
from tensorflow.keras.models import Sequential, load_model
from tensorflow.keras.callbacks import ModelCheckpoint,  EarlyStopping
from tensorflow.keras.layers import Conv1D, MaxPooling1D, Dense, Dropout, Flatten, BatchNormalization
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
from tensorflow.keras.optimizers import Adam
from sklearn.preprocessing import LabelEncoder
import h5py
from sklearn.utils.class_weight import compute_class_weight
from autolrtuner import AutoLRTuner
import os






# Mount Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


#Main

## Set up HDF5 File and labeling Data

In [ ]:
HDF5_file_path = "/content/drive/MyDrive/Qeej_Hmong_Dataset/Save/HDF5s"
HDF5_dict = {}

if os.path.exists(HDF5_file_path):
  for root, dirs, files in os.walk(HDF5_file_path):
      for file in files:
          if file.endswith(".h5"):
             data_set_path = os.path.join(root, file)
             if os.path.exists(data_set_path):
                print("Retrieving data from drive with path:", data_set_path)
                with h5py.File(data_set_path, "r") as f:
                  X = f["features"][:]
                  y = f["labels"][:]
                  HDF5_dict[data_set_path] = (X, y)



Retrieving data from drive with path: /content/drive/MyDrive/Qeej_Hmong_Dataset/Save/HDF5s/Qeej_Hmong_Features_With_Data_Au(repeat = 1).h5
Retrieving data from drive with path: /content/drive/MyDrive/Qeej_Hmong_Dataset/Save/HDF5s/Qeej_Hmong_Features_Without_DataAu.h5
Retrieving data from drive with path: /content/drive/MyDrive/Qeej_Hmong_Dataset/Save/HDF5s/Qeej_Hmong_Features_With_Data_Au(repeat = 2).h5


## Assess Model Learning rate per HDF5

### Preparing Data

In [ ]:
def get_tranning_data(X, y):
    #label encoding
    le = LabelEncoder()
    y_int = le.fit_transform(y)
    y_cat = to_categorical(y_int)

    class_names = le.classes_

    #set up training, testing, validation data
    X_train, X_temp, y_train, y_temp = train_test_split(
    X, y_cat, test_size=0.3, stratify=y_int)

    X_val, X_test, y_val, y_test = train_test_split(
        X_temp, y_temp, test_size=0.1)

    y_train_int = np.argmax(y_train, axis=1)
    print("check y train int:", y_train_int)
    return X_train, y_train, y_cat


### Model

In [ ]:


def build_model(y_cat):
    model = Sequential([
        Conv1D(32, kernel_size=5, activation='relu',
              input_shape=(X.shape[1], 1)),
        MaxPooling1D(2),
        Dropout(0.25),

        Conv1D(64, kernel_size=5, activation='relu'),
        MaxPooling1D(2),
        Dropout(0.25),

        Flatten(),
        Dense(128, activation='relu'),
        Dropout(0.5),

        Dense(y_cat.shape[1], activation='softmax')
    ])

    optimizer = Adam(learning_rate=0.001)

    model.compile(
        optimizer=optimizer,
        loss=CategoricalCrossentropy(),
        metrics=['accuracy']
    )

    return model

def asssess_lr_models(model, X_train, y_train, batch_size = 512, num_evals=200):
      tuner = AutoLRTuner(model)
      tuner.Tune(
          X_train,
          y_train,
          batch_size=batch_size,
          num_evals=num_evals
      )

      best_lr = tuner.GetBestLR()

      return best_lr



In [ ]:
lr_dict = {}
batch_size_list = [128,256,512,1024]
for HDF5_file, data  in HDF5_dict.items():
  for batch_size in batch_size_list:
    X, y = data
    X_train, y_train, y_cat = get_tranning_data(X, y)
    model = build_model(y_cat)
    best_lr = asssess_lr_models(model, X_train, y_train)
    key = HDF5_file + "_" + str(batch_size)
    lr_dict[key] = best_lr
print(lr_dict)




check y train int: [12 41 59 ... 42 50 37]


/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
100%|██████████| 100/100 [22:22<00:00, 13.43s/it]


check y train int: [56  0 42 ... 36 28 21]


/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
100%|██████████| 100/100 [23:29<00:00, 14.09s/it]


check y train int: [56  0 42 ... 36 28 21]


...

check y train int: [17 38 28 ... 28 31 48]


/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
100%|██████████| 100/100 [22:43<00:00, 13.63s/it]


check y train int: [17 38 28 ... 28 31 48]


/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
100%|██████████| 100/100 [24:57<00:00, 14.98s/it]

{'/content/drive/MyDrive/Qeej_Hmong_Dataset/Save/HDF5s/Qeej_Hmong_Features_With_Data_Au(repeat = 1).h5_128': 0.010050261155778895, '/content/drive/MyDrive/Qeej_Hmong_Dataset/Save/HDF5s/Qeej_Hmong_Features_With_Data_Au(repeat = 1).h5_256': 0.005025135577889447, '/content/drive/MyDrive/Qeej_Hmong_Dataset/Save/HDF5s/Qeej_Hmong_Features_With_Data_Au(repeat = 1).h5_512': 0.010050261155778895, '/content/drive/MyDrive/Qeej_Hmong_Dataset/Save/HDF5s/Qeej_Hmong_Features_With_Data_Au(repeat = 1).h5_1024': 0.005025135577889447, '/content/drive/MyDrive/Qeej_Hmong_Dataset/Save/HDF5s/Qeej_Hmong_Features_Without_DataAu.h5_128': 0.005025135577889447, '/content/drive/MyDrive/Qeej_Hmong_Dataset/Save/HDF5s/Qeej_Hmong_Features_Without_DataAu.h5_256': 0.005025135577889447, '/content/drive/MyDrive/Qeej_Hmong_Dataset/Save/HDF5s/Qeej_Hmong_Features_Without_DataAu.h5_512': 0.005025135577889447, '/content/drive/MyDrive/Qeej_Hmong_Dataset/Save/HDF5s/Qeej_Hmong_Features_Without_DataAu.h5_1024': 0.00502513557788944